<a href="https://colab.research.google.com/github/SamDerden/Fraud_Detection_with-XAI_and_LLM/blob/main/%D0%A1%D0%B5%D0%BC%D0%B8%D0%BD%D0%B0%D1%80_11_12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Антифрод-сервис банка (FastAPI)

## 1. Установка пакетов

In [ ]:
!pip install fastapi uvicorn scikit-learn pydantic nest_asyncio -q
print("Пакеты установлены")

Пакеты установлены


## 2. Код сервиса (модель + FastAPI)

In [ ]:
from __future__ import annotations

from datetime import datetime
from typing import Literal, Optional

import numpy as np
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field, field_validator
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

MODEL_VERSION = "2026-09-30"
FEATURE_NAMES = [
    "amount_log", "hour", "day_of_week", "tx_count_24h", "tx_count_7d",
    "distinct_merchants_7d", "distinct_countries_30d", "device_age_days",
    "is_new_device", "is_night_tx", "avg_amount_ratio", "channel_mobile",
]


def _train_model():
    X, y = make_classification(
        n_samples=4000, n_features=12, n_informative=8, n_redundant=2,
        weights=[0.85, 0.15], flip_y=0.02, random_state=42,
    )
    X_train, _, y_train, _ = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    m = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    m.fit(X_train, y_train)
    return m


model = _train_model()


# ---------- Pydantic-схемы (контракт) ----------
class Geo(BaseModel):
    country: Optional[str] = Field(None, examples=["RU"])
    city: Optional[str] = Field(None, examples=["Kazan"])


class ScoreRequest(BaseModel):
    transaction_id: str = Field(..., examples=["txn_839201"])
    user_id: str = Field(..., examples=["usr_10482"])
    amount: float = Field(..., gt=0, examples=[15700.50])
    currency: str = Field("RUB", examples=["RUB"])
    merchant_id: Optional[str] = Field(None, examples=["mrc_5521"])
    device_id: Optional[str] = Field(None, examples=["dev_9f3a1c"])
    geo: Optional[Geo] = None
    timestamp: str = Field(..., examples=["2026-09-14T03:32:00Z"])
    hour: Optional[int] = Field(None, ge=0, le=23)
    day_of_week: Optional[int] = Field(None, ge=0, le=6)
    tx_count_24h: Optional[int] = Field(0, ge=0)
    tx_count_7d: Optional[int] = Field(0, ge=0)
    distinct_merchants_7d: Optional[int] = Field(1, ge=1)
    distinct_countries_30d: Optional[int] = Field(1, ge=1)
    device_age_days: Optional[int] = Field(100, ge=0)
    channel: Optional[str] = Field("mobile", examples=["mobile", "web", "pos", "api"])

    @field_validator("timestamp")
    @classmethod
    def timestamp_not_empty(cls, v: str) -> str:
        if not v or not v.strip():
            raise ValueError("timestamp не может быть пустым")
        return v


class ScoreResponse(BaseModel):
    risk_score: float = Field(..., ge=0.0, le=1.0)
    decision: Literal["approve", "review", "block"]
    explanation: str
    model_version: str
    transaction_id: str


class HealthResponse(BaseModel):
    status: str
    model_version: str
    timestamp: str


class ModelInfoResponse(BaseModel):
    model_version: str
    model_type: str
    n_features: int
    feature_names: list[str]
    decisions: list[str]


class ServiceInfoResponse(BaseModel):
    service: str
    version: str
    endpoints: list[str]
    docs: str


# ---------- Логика ----------
def _build_features(req: ScoreRequest) -> np.ndarray:
    try:
        ts = datetime.fromisoformat(req.timestamp.replace("Z", "+00:00"))
        hour = req.hour if req.hour is not None else ts.hour
        dow = req.day_of_week if req.day_of_week is not None else ts.weekday()
    except Exception:
        hour = req.hour if req.hour is not None else 12
        dow = req.day_of_week if req.day_of_week is not None else 2

    is_new_device = 1 if (req.device_age_days or 100) < 7 else 0
    is_night = 1 if hour <= 5 else 0
    channel_mobile = 1 if (req.channel or "mobile") == "mobile" else 0

    features = np.array([
        np.log1p(req.amount), hour, dow,
        req.tx_count_24h or 0, req.tx_count_7d or 0,
        req.distinct_merchants_7d or 1, req.distinct_countries_30d or 1,
        req.device_age_days or 100, is_new_device, is_night, 1.0, channel_mobile,
    ], dtype=float)
    return features.reshape(1, -1)


def _make_decision(risk_score: float) -> str:
    if risk_score >= 0.75:
        return "block"
    if risk_score >= 0.45:
        return "review"
    return "approve"


def _make_explanation(req: ScoreRequest, risk_score: float, decision: str) -> str:
    reasons = []
    if req.amount >= 10000:
        reasons.append("крупная сумма")
    hour = req.hour
    if hour is None:
        try:
            hour = datetime.fromisoformat(req.timestamp.replace("Z", "+00:00")).hour
        except Exception:
            hour = 12
    if hour <= 5:
        reasons.append("ночное время")
    if (req.device_age_days or 100) < 7:
        reasons.append("новое устройство")
    if (req.tx_count_24h or 0) >= 8:
        reasons.append("много операций за 24ч")
    if (req.distinct_countries_30d or 1) >= 3:
        reasons.append("несколько стран за 30 дней")
    if not reasons:
        if decision == "approve":
            return "Признаки соответствуют обычному поведению клиента"
        return f"Риск {risk_score:.2f} — требуется проверка"
    return " + ".join(reasons)


# ---------- Приложение ----------
app = FastAPI(
    title="Антифрод-сервис банка",
    description="ML-сервис детекции подозрительных транзакций",
    version="1.0.0",
)


@app.get("/", response_model=ServiceInfoResponse, tags=["service"])
def root():
    return ServiceInfoResponse(
        service="antifraud-scoring",
        version="1.0.0",
        endpoints=["/", "/health", "/v1/score", "/v1/model-info", "/docs"],
        docs="/docs",
    )


@app.get("/health", response_model=HealthResponse, tags=["service"])
def health():
    if model is None:
        raise HTTPException(status_code=503, detail="Модель не загружена")
    return HealthResponse(
        status="ok",
        model_version=MODEL_VERSION,
        timestamp=datetime.utcnow().isoformat() + "Z",
    )


@app.post("/v1/score", response_model=ScoreResponse, tags=["scoring"])
def score_transaction(req: ScoreRequest):
    if model is None:
        raise HTTPException(status_code=503, detail="Сервис временно недоступен")
    features = _build_features(req)
    proba = float(model.predict_proba(features)[0, 1])
    decision = _make_decision(proba)
    explanation = _make_explanation(req, proba, decision)
    return ScoreResponse(
        risk_score=round(proba, 4),
        decision=decision,
        explanation=explanation,
        model_version=MODEL_VERSION,
        transaction_id=req.transaction_id,
    )


@app.get("/v1/model-info", response_model=ModelInfoResponse, tags=["scoring"])
def model_info():
    return ModelInfoResponse(
        model_version=MODEL_VERSION,
        model_type="RandomForestClassifier",
        n_features=len(FEATURE_NAMES),
        feature_names=FEATURE_NAMES,
        decisions=["approve", "review", "block"],
    )


print("Приложение создано, модель обучена.")
print("Эндпоинты: /, /health, /v1/score, /v1/model-info")

Приложение создано, модель обучена.
Эндпоинты: /, /health, /v1/score, /v1/model-info


## 3. Проверка через TestClient

In [ ]:
from fastapi.testclient import TestClient

client = TestClient(app)

print("=" * 60)
print("1. GET /")
print("=" * 60)
r = client.get("/")
print("status:", r.status_code)
print(r.json())

print("\n" + "=" * 60)
print("2. GET /health")
print("=" * 60)
r = client.get("/health")
print("status:", r.status_code)
print(r.json())

print("\n" + "=" * 60)
print("3. GET /v1/model-info")
print("=" * 60)
r = client.get("/v1/model-info")
print("status:", r.status_code)
print(r.json())

print("\n" + "=" * 60)
print("4. POST /v1/score — подозрительная транзакция")
print("=" * 60)
payload = {
    "transaction_id": "txn_839201",
    "user_id": "usr_10482",
    "amount": 15700.50,
    "currency": "RUB",
    "merchant_id": "mrc_5521",
    "device_id": "dev_9f3a1c",
    "geo": {"country": "RU", "city": "Kazan"},
    "timestamp": "2026-09-14T03:32:00Z",
    "tx_count_24h": 2,
    "device_age_days": 3,
    "channel": "mobile",
}
r = client.post("/v1/score", json=payload)
print("status:", r.status_code)
print(r.json())

print("\n" + "=" * 60)
print("5. POST /v1/score — обычная транзакция")
print("=" * 60)
payload2 = {
    "transaction_id": "txn_100",
    "user_id": "usr_1",
    "amount": 500.0,
    "timestamp": "2026-09-14T14:00:00Z",
    "device_age_days": 200,
    "tx_count_24h": 1,
}
r = client.post("/v1/score", json=payload2)
print("status:", r.status_code)
print(r.json())

print("\n" + "=" * 60)
print("6. Валидация — ожидаем 422 (amount < 0)")
print("=" * 60)
r = client.post("/v1/score", json={
    "transaction_id": "txn_bad",
    "user_id": "usr_1",
    "amount": -10,
    "timestamp": "2026-09-14T14:00:00Z",
})
print("status:", r.status_code)
print("(ожидается 422)")

1. GET /
status: 200
{'service': 'antifraud-scoring', 'version': '1.0.0', 'endpoints': ['/', '/health', '/v1/score', '/v1/model-info', '/docs'], 'docs': '/docs'}

2. GET /health
status: 200
{'status': 'ok', 'model_version': '2026-09-30', 'timestamp': '2026-10-26T17:50:49.430787Z'}

3. GET /v1/model-info
status: 200
{'model_version': '2026-09-30', 'model_type': 'RandomForestClassifier', 'n_features': 12, 'feature_names': ['amount_log', 'hour', 'day_of_week', 'tx_count_24h', 'tx_count_7d', 'distinct_merchants_7d', 'distinct_countries_30d', 'device_age_days', 'is_new_device', 'is_night_tx', 'avg_amount_ratio', 'channel_mobile'], 'decisions': ['approve', 'review', 'block']}

4. POST /v1/score — подозрительная транзакция
status: 200
{'risk_score': 0.21, 'decision': 'approve', 'explanation': 'крупная сумма + ночное время + новое устройство', 'model_version': '2026-09-30', 'transaction_id': 'txn_839201'}

5. POST /v1/score — обычная транзакция
status: 200
{'risk_score': 0.3, 'decision': 'appr

/tmp/ipykernel_3311/1731885873.py:179: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp=datetime.utcnow().isoformat() + "Z",


Swagger UI в браузере

In [ ]:
import nest_asyncio
import uvicorn
import threading
import time
import socket

nest_asyncio.apply()

def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

# если порт занят предыдущим запуском — игнорируем
threading.Thread(target=run_server, daemon=True).start()
time.sleep(3)

# проверка, что порт слушает
s = socket.socket()
try:
    s.connect(("127.0.0.1", 8000))
    print("OK: uvicorn слушает 8000")
except Exception as e:
    print("Ошибка: сервер не поднялся:", e)
finally:
    s.close()

OK: uvicorn слушает 8000


cloudflared (без аккаунта и токена)

In [ ]:
import subprocess, re, time, os

# скачать binary один раз
if not os.path.exists("cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared

# убить старый процесс, если был
!pkill -f cloudflared || true
time.sleep(1)

# запуск в фоне, логи в файл
log = open("cf.log", "w")
proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=log, stderr=subprocess.STDOUT,
)

# ждём URL в логе (до ~20 сек)
url = None
for _ in range(40):
    time.sleep(0.5)
    text = open("cf.log").read()
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", text)
    if m:
        url = m.group(0)
        break

if url:
    print("Swagger UI →", url + "/docs")
    print("Health     →", url + "/health")
else:
    print("URL не найден. Лог:")
    print(open("cf.log").read()[-2000:])

^C
Swagger UI → https://vids-greatest-douglas-players.trycloudflare.com/docs
Health     → https://vids-greatest-douglas-players.trycloudflare.com/health
^C
Swagger UI → https://imaging-ist-ashley-builds.trycloudflare.com/docs
Health     → https://imaging-ist-ashley-builds.trycloudflare.com/health
